# Testing `project()` — CIFAR-10 Embedding Projections

Download CIFAR-10, extract embeddings with ResNet50 via `OnnxExtractor`, and visualize with `project()`.

## 1. Setup — Download model and dataset

In [ ]:
import os

import cv2
import numpy as np
import requests
from torchvision.datasets import CIFAR10

from dataeval_plots import project


def download_onnx_model(url: str, save_path: str) -> None:
    """Download ONNX model if not already cached."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    if os.path.exists(save_path):
        print(f"Model already exists at {save_path}")
        return
    print(f"Downloading model from {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete.")


model_url = "https://github.com/onnx/models/raw/main/validated/vision/classification/resnet/model/resnet50-v2-7.onnx"
model_path = "data/resnet50-v2-7.onnx"
download_onnx_model(model_url, model_path)

In [ ]:
from dataeval.utils.onnx import to_encoding_model

encoding_model, embedding_layer = to_encoding_model(model_path)
print(f"Embedding layer: {embedding_layer}")
print(f"In-memory model size: {len(encoding_model):,} bytes")

## 2. Load CIFAR-10 and create a MAITE-compatible wrapper

In [ ]:
CIFAR10_CLASSES = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck",
}


class CIFAR10Dataset:
    """Thin MAITE-compatible wrapper around torchvision CIFAR10."""

    def __init__(self, root: str = "./data", train: bool = False, limit: int | None = None):
        self._ds = CIFAR10(root=root, train=train, download=True)
        self._limit = min(limit, len(self._ds)) if limit else len(self._ds)

    @property
    def metadata(self) -> dict:
        return {"id": "cifar10", "index2label": CIFAR10_CLASSES}

    def __len__(self) -> int:
        return self._limit

    def __getitem__(self, index: int) -> tuple[np.ndarray, int, dict]:
        img_pil, label = self._ds[index]
        # Convert PIL -> CHW float32 numpy
        img = np.array(img_pil, dtype=np.float32).transpose(2, 0, 1)  # HWC -> CHW
        return img, label, {}


dataset = CIFAR10Dataset(limit=1000)
print(f"Dataset size: {len(dataset)}")
print(f"Image shape: {dataset[0][0].shape}")  # (3, 32, 32)

## 3. Extract embeddings with OnnxExtractor + Embeddings

In [ ]:
from dataeval import Embeddings
from dataeval.extractors import OnnxExtractor


def preprocess(image: np.ndarray) -> np.ndarray:
    """Preprocess for ResNet50: resize 32x32 -> 224x224, normalize with ImageNet stats."""
    hwc = image.transpose(1, 2, 0)  # CHW -> HWC
    resized = cv2.resize(hwc, (224, 224))
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    normalized = (resized / 255.0 - mean) / std
    chw = normalized.transpose(2, 0, 1)  # HWC -> CHW
    return chw


extractor = OnnxExtractor(
    model=encoding_model,
    transforms=preprocess,
    output_name=embedding_layer,
)

embeddings = Embeddings(dataset, extractor=extractor, batch_size=32)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Extract class labels for coloring
class_labels = np.array([dataset[i][1] for i in range(len(dataset))])
print(f"Labels shape: {class_labels.shape}")
print(f"Unique classes: {np.unique(class_labels)}")

## 4. Single method — PCA

In [ ]:
fig = project(
    embeddings,
    method="pca",
    labels=class_labels,
    label_names=CIFAR10_CLASSES,
    figsize=(10, 8),
)

## 5. Single method — t-SNE

In [ ]:
fig = project(
    embeddings,
    method="tsne",
    labels=class_labels,
    label_names=CIFAR10_CLASSES,
    figsize=(10, 8),
    perplexity=30.0,
)

## 6. Compare multiple methods — grid

In [ ]:
fig = project(
    embeddings,
    method=["isomap", "mds", "spectral", "umap", "pacmap", "phate"],
    labels=class_labels,
    label_names=CIFAR10_CLASSES,
    figsize=(18, 10),
)

## 7. 3D projection — PCA

In [ ]:
fig = project(
    embeddings,
    method="pca",
    dimensions=3,
    labels=class_labels,
    label_names=CIFAR10_CLASSES,
    figsize=(10, 8),
)

## 7a. 3D projection — t-SNE with Plotly (interactive)

In [ ]:
fig = project(
    embeddings,
    method="tsne",
    dimensions=3,
    labels=class_labels,
    label_names=CIFAR10_CLASSES,
    figsize=(12, 10),
    backend="plotly",
)
fig.show()

## 7b. 3D projection — multiple methods with Plotly (interactive)

In [ ]:
fig = project(
    embeddings,
    method=["pca", "tsne", "isomap", "mds", "spectral", "truncated_svd"],
    dimensions=3,
    labels=class_labels,
    label_names=CIFAR10_CLASSES,
    figsize=(12, 10),
    backend="plotly",
)
fig.show()

## 8. No labels — unsupervised view

In [ ]:
fig = project(
    embeddings,
    method="tsne",
    figsize=(10, 8),
    title="t-SNE (no labels)",
)

## 9. Tall grid — aspect ratio heuristic

A tall figsize should produce more rows than columns.

In [ ]:
fig = project(
    embeddings,
    method=["pca", "tsne", "isomap", "mds"],
    labels=class_labels,
    label_names=CIFAR10_CLASSES,
    figsize=(8, 16),  # tall → should stack vertically
)

## 10. Pre-reduced embeddings (`method=None`)

In [ ]:
from sklearn.decomposition import PCA

# Manually reduce, then plot with method=None
pca = PCA(n_components=2)
reduced = pca.fit_transform(np.asarray(embeddings))

fig = project(
    reduced,
    method=None,
    labels=class_labels,
    label_names=CIFAR10_CLASSES,
    figsize=(10, 8),
    title="Pre-reduced PCA (method=None)",
)